In [ ]:
# ================================
# Q1 MEDICAL JOURNAL FULL PIPELINE
# RetinaViT Training + External Validation
# ================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import copy
import pandas as pd

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import *
from sklearn.calibration import calibration_curve
from scipy.stats import ttest_rel

# ================= CONFIG =================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_A = r"D:\Datasets\ODIR5K"
DATASET_B = r"D:\Datasets\DDR_APTOS_MESSIDOR"
MODEL_PATH = r"D:\Models\RetinaViT_Backbone.pth"

SAVE_DIR = r"D:\Q1_RESULTS"
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 5
LR = 1e-4

# ================= TRANSFORMS =================
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# ================= LOAD DATA =================
train_full = datasets.ImageFolder(f"{DATASET_A}/train", transform=train_transform)
external_test = datasets.ImageFolder(f"{DATASET_B}/test", transform=test_transform)

val_size = int(0.2 * len(train_full))
train_size = len(train_full) - val_size
train_ds, val_ds = random_split(train_full, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
external_loader = DataLoader(external_test, batch_size=BATCH_SIZE, shuffle=False)

# ================= LOAD MODEL =================
model = torch.load(MODEL_PATH)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# ================= TRAINING =================
best_val_acc = 0
best_weights = copy.deepcopy(model.state_dict())

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    model.eval()
    val_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = torch.argmax(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = copy.deepcopy(model.state_dict())

torch.save(best_weights, f"{SAVE_DIR}/best_model.pth")

# ================= EVALUATION FUNCTION =================
def evaluate(loader):
    model.load_state_dict(best_weights)
    model.eval()

    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# ================= METRICS =================
def compute_metrics(labels, preds, probs, name):

    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='weighted')
    recall = recall_score(labels, preds, average='weighted')
    f1 = f1_score(labels, preds, average='weighted')
    auc = roc_auc_score(labels, probs, multi_class='ovr')

    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"{name} Confusion Matrix")
    plt.savefig(f"{SAVE_DIR}/{name}_CM.png")
    plt.close()

    print(f"\n{name} Results")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"AUC: {auc:.4f}")

    return acc

# ================= CALIBRATION =================
def compute_ece(probs, labels, n_bins=15):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    bins = np.linspace(0,1,n_bins+1)
    ece = 0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        if np.sum(mask) > 0:
            acc_bin = np.mean(predictions[mask] == labels[mask])
            conf_bin = np.mean(confidences[mask])
            ece += np.abs(acc_bin-conf_bin)*np.sum(mask)/len(labels)
    return ece

# ================= BOOTSTRAP CI =================
def bootstrap_ci(labels, preds, n_bootstrap=1000):
    scores = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(labels), len(labels), replace=True)
        scores.append(accuracy_score(labels[idx], preds[idx]))
    return np.percentile(scores,2.5), np.percentile(scores,97.5)

# ================= MC DROPOUT =================
def mc_dropout(loader, passes=20):
    model.train()
    variances = []

    for images, _ in loader:
        images = images.to(DEVICE)
        preds = []

        for _ in range(passes):
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            preds.append(probs.detach().cpu().numpy())

        preds = np.stack(preds)
        variance = np.var(preds, axis=0)
        variances.extend(np.mean(variance, axis=1))

    return np.array(variances)

# ================= RUN INTERNAL & EXTERNAL =================
int_labels, int_preds, int_probs = evaluate(val_loader)
ext_labels, ext_preds, ext_probs = evaluate(external_loader)

acc_internal = compute_metrics(int_labels, int_preds, int_probs, "Internal")
acc_external = compute_metrics(ext_labels, ext_preds, ext_probs, "External")

ece_external = compute_ece(ext_probs, ext_labels)
print("External ECE:", ece_external)

ci_low, ci_high = bootstrap_ci(ext_labels, ext_preds)
print("External 95% CI:", ci_low, "-", ci_high)

# Statistical comparison
t_stat, p_val = ttest_rel(int_preds[:len(ext_preds)], ext_preds)
print("Paired t-test p-value:", p_val)

effect_size = (acc_internal - acc_external) / np.std([acc_internal, acc_external])
print("Effect Size (Cohen's d):", effect_size)

# Uncertainty
uncertainty = mc_dropout(external_loader)
plt.hist(uncertainty, bins=30)
plt.title("Uncertainty Distribution")
plt.savefig(f"{SAVE_DIR}/Uncertainty.png")
plt.close()

# Save curves
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.savefig(f"{SAVE_DIR}/Loss_Curve.png")
plt.close()

plt.plot(train_accs, label="Train Acc")
plt.plot(val_accs, label="Val Acc")
plt.legend()
plt.savefig(f"{SAVE_DIR}/Accuracy_Curve.png")
plt.close()




================ TRAINING STARTED ================

Epoch 1/50
Train Loss: 1.6350 | Train Acc: 0.5546
Val Loss: 1.7875 | Val Acc: 0.5030

Epoch 2/50
Train Loss: 1.5231 | Train Acc: 0.5492
Val Loss: 1.6306 | Val Acc: 0.5186

Epoch 3/50
Train Loss: 1.4443 | Train Acc: 0.5564
Val Loss: 1.5491 | Val Acc: 0.5293

Epoch 4/50
Train Loss: 1.3633 | Train Acc: 0.5676
Val Loss: 1.4227 | Val Acc: 0.5133

Epoch 5/50
Train Loss: 1.2577 | Train Acc: 0.5806
Val Loss: 1.3437 | Val Acc: 0.5241

Epoch 6/50
Train Loss: 1.1909 | Train Acc: 0.5858
Val Loss: 1.2364 | Val Acc: 0.5340

Epoch 7/50
Train Loss: 1.1108 | Train Acc: 0.5920
Val Loss: 1.1867 | Val Acc: 0.5460

Epoch 8/50
Train Loss: 1.0470 | Train Acc: 0.6082
Val Loss: 1.0734 | Val Acc: 0.5433

Epoch 9/50
Train Loss: 0.9612 | Train Acc: 0.6233
Val Loss: 1.0475 | Val Acc: 0.5701

Epoch 10/50
Train Loss: 0.9103 | Train Acc: 0.6257
Val Loss: 0.9387 | Val Acc: 0.5666

Epoch 11/50
Train Loss: 0.8463 | Train Acc: 0.6207
Val Loss: 0.8970 | Val Acc: 0.5883
